# Schirmer aperture E/B maps (binned grid, DP2 tract 9813)

Alex Broughton · April 2026. Rubin stack and Butler access required.

This notebook mirrors `python/schirmer_snr_weight.py` (Shenming Fu; there is no separate `schirmer_snr.py` in this repo): **2D binning** of sources, then for each grid cell a Schirmer-filtered sum of tangential/cross ellipticities with the **original Schirmer noise** denominator from that script. Data paths and selection follow `dp2_tract9813_metadetect_massmap.ipynb`, but the geometry is a **flat tangent plane** in tract-native virtual pixels (from RA/Dec at `PIX_SCALE_ARCSEC`), not HEALPix.


## Introduction

- **Binning:** `scipy.stats.binned_statistic_2d` weighted means for `gauss_g1`, `gauss_g2`, and weighted `(g1^2+g2^2)` with `weight**2` for the noise term, matching `get_bin_stat_weight` in `schirmer_snr_weight.py`.
- **Aperture:** For each output bin `(row, col)`, sum over the grid `Q(r) \times e_t` and `Q(r) \times e_x` with Fu’s \(e_t\), \(e_x\) from `atan2(dy, dx)` on bin indices; \(n_{M_{\rm ap}} = \sqrt{\sum Q^2 e_{\rm sq}} / \sqrt{2}\).
- **S/N maps:** `M_{\rm ap,E}/n_{M_{\rm ap}}` and `M_{\rm ap,B}/n_{M_{\rm ap}}` as in the script’s `plot_E_B(..., "M_ap_SNR")`.
- **Coordinates:** Catalog positions are mapped to virtual pixels \((x,y)\) from RA/Dec relative to the tract median, so you do not need catalog `x`/`y` columns.


## 1.0 Setup


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from tqdm.auto import tqdm
from lsst.daf.butler import Butler

CONFIG = {
    "REPO": "/sdf/data/rubin/repo/dp2_prep",
    "COLLECTION": "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "SKYMAP": "lsst_cells_v2",
    "INSTRUMENT": "LSSTCam",
    "TRACT": 9813,
    "RS_INPUT_PIX": 10000,  # Schirmer scale in virtual pixels (same convention as HEALPix notebook)
    "PIX_SCALE_ARCSEC": 0.2,
    "BIN_SIZE_NATIVE_PIX": 32.0,  # Bin width in virtual pixels (matches schirmer_snr_weight.py default)
}

CONFIG["BIN_SIZE_ARCSEC"] = float(CONFIG["BIN_SIZE_NATIVE_PIX"]) * float(CONFIG["PIX_SCALE_ARCSEC"])

%matplotlib inline
plt.rcParams.update({"font.size": 12})

butler = Butler(
    CONFIG["REPO"],
    collections=[CONFIG["COLLECTION"]],
    skymap=CONFIG["SKYMAP"],
    instrument=CONFIG["INSTRUMENT"],
)

print("TRACT", CONFIG["TRACT"])
print("Rs (virtual pix)", CONFIG["RS_INPUT_PIX"], "| bin (virtual pix)", CONFIG["BIN_SIZE_NATIVE_PIX"])
print("Rs / bin =", CONFIG["RS_INPUT_PIX"] / CONFIG["BIN_SIZE_NATIVE_PIX"], "(Schirmer Rs in bin cells)")


## 2.0 Core functions (from `schirmer_snr_weight.py`)


In [ ]:
def Schirmer_weight(r, Rs):
    """Schirmer Q(r/Rs); r and Rs in the same length units (here: bin index)."""
    r = np.asarray(r, dtype=np.float64)
    Rs = float(Rs)
    x = r / Rs
    a, b, c, d, xc = 6.0, 150.0, 47.0, 50.0, 0.15
    Q = 1.0 / (1.0 + np.exp(a - b * x) + np.exp(d * x - c))
    ratio = x / xc
    with np.errstate(divide="ignore", invalid="ignore"):
        inner = np.tanh(ratio) / ratio
    inner = np.where(np.abs(ratio) < 1e-14, 1.0, inner)
    Q = Q * inner
    return np.where(np.isfinite(Q), Q, 0.0)


def get_bin_stat_weight(x, y, arr, weight, x_bin, y_bin):
    """Weighted mean on 2D bins; transpose so rows ↔ y (Fu script)."""
    statistic_0, _, _, _ = stats.binned_statistic_2d(
        x,
        y,
        arr * weight,
        statistic="sum",
        bins=[x_bin, y_bin],
    )
    statistic_1, _, _, _ = stats.binned_statistic_2d(
        x,
        y,
        weight,
        statistic="sum",
        bins=[x_bin, y_bin],
    )
    with np.errstate(divide="ignore", invalid="ignore"):
        out = statistic_0 / statistic_1
    return out.T


def compute_M_ap_at_pixel(ind, xv, yv, e1_binned, e2_binned, e_sq_binned, Rs):
    """One grid cell; matches schirmer_snr_weight.compute_M_ap_at_pixel logic."""
    row, col = int(ind[0]), int(ind[1])
    Qw = Schirmer_weight(((xv - col) ** 2 + (yv - row) ** 2) ** 0.5, Rs)
    dx, dy = xv - col, yv - row
    angle = np.arctan2(dy, dx)
    et = -e1_binned * np.cos(2.0 * angle) - e2_binned * np.sin(2.0 * angle)
    ex = +e1_binned * np.sin(2.0 * angle) - e2_binned * np.cos(2.0 * angle)
    M_ap_E_tmp = np.nansum(Qw * et)
    M_ap_B_tmp = np.nansum(Qw * ex)
    tmp = (Qw**2) * e_sq_binned
    n_M_ap_tmp = np.sqrt(np.nansum(tmp)) / np.sqrt(2.0)
    return M_ap_E_tmp, M_ap_B_tmp, n_M_ap_tmp


## 3.0 Load data and selection


In [ ]:
data = butler.get("object_shear_all", dataId={"tract": CONFIG["TRACT"]})
print(f"Rows: {len(data):,}")


In [ ]:
mask = data["metaStep"] == "ns"
mask &= data["image_flags"] == 0
mask &= data["psfOriginal_flags"] == 0
mask &= data["bmask_flags"] == 0
mask &= data["ormask_flags"] == 0
mask &= data["mfrac"] < 0.1

shear_catalog = data[mask]
print("Sources before quality cuts:", len(data))
print("Sources after quality cuts:", len(shear_catalog))


## 4.0 Virtual pixel coordinates and weights

Positions \((x,y)\) in the same virtual-pixel units as `RS_INPUT_PIX` and `BIN_SIZE_NATIVE_PIX`: one pixel = `PIX_SCALE_ARCSEC` on the tangent plane at the tract median declination.


In [ ]:
def _col(tbl, name):
    return np.array(tbl[name]).astype(np.float64, copy=False)


ra_deg = _col(shear_catalog, "ra")
dec_deg = _col(shear_catalog, "dec")
e1 = _col(shear_catalog, "gauss_g1")
e2 = _col(shear_catalog, "gauss_g2")

_var_sum = _col(shear_catalog, "gauss_g1_g1_Cov") + _col(shear_catalog, "gauss_g2_g2_Cov")
with np.errstate(divide="ignore", invalid="ignore"):
    weight = np.where(np.isfinite(_var_sum) & (_var_sum > 0.0), 1.0 / _var_sum, 0.0)

ra0 = float(np.median(ra_deg))
dec0 = float(np.median(dec_deg))
cos_dec0 = np.cos(np.deg2rad(dec0))
ps = float(CONFIG["PIX_SCALE_ARCSEC"])
x = (ra_deg - ra0) * cos_dec0 * (3600.0 / ps)
y = (dec_deg - dec0) * (3600.0 / ps)

mfinite = np.isfinite(x) & np.isfinite(y) & np.isfinite(e1) & np.isfinite(e2) & (weight > 0)
x, y, e1, e2, weight = x[mfinite], y[mfinite], e1[mfinite], e2[mfinite], weight[mfinite]

print("N galaxies used:", x.size)


## 5.0 Binning and aperture mass


In [ ]:
bin_size = float(CONFIG["BIN_SIZE_NATIVE_PIX"])
Rs_input = float(CONFIG["RS_INPUT_PIX"])

x_min = float(np.min(x))
x_max = float(np.max(x))
y_min = float(np.min(y))
y_max = float(np.max(y))

ncol = int(np.ceil((x_max - x_min) / bin_size))
nrow = int(np.ceil((y_max - y_min) / bin_size))
x_bin = x_min + np.arange(ncol + 1) * bin_size
y_bin = y_min + np.arange(nrow + 1) * bin_size

print("nrow, ncol:", nrow, ncol)

xv, yv = np.meshgrid(np.arange(ncol), np.arange(nrow))
coord_list = list(zip(yv.ravel(), xv.ravel()))

Rs = Rs_input / bin_size

print("Running bin stat...")
e1_binned = get_bin_stat_weight(x, y, e1, weight, x_bin, y_bin)
e2_binned = get_bin_stat_weight(x, y, e2, weight, x_bin, y_bin)
e_sq_binned = get_bin_stat_weight(x, y, e1**2 + e2**2, weight**2, x_bin, y_bin)

print("Computing aperture mass...")
result = [
    compute_M_ap_at_pixel(ind, xv, yv, e1_binned, e2_binned, e_sq_binned, Rs)
    for ind in tqdm(coord_list, desc="Aperture cells")
]
result = np.asarray(result, dtype=np.float64)
M_ap_E = result[:, 0].reshape((nrow, ncol))
M_ap_B = result[:, 1].reshape((nrow, ncol))
n_M_ap = result[:, 2].reshape((nrow, ncol))

with np.errstate(divide="ignore", invalid="ignore"):
    SNR_E = M_ap_E / n_M_ap
    SNR_B = M_ap_B / n_M_ap

print("Done.")


## 6.0 Figures (E/B S/N like `plot_E_B`)


In [ ]:
def _sym_lim(mat, pct=95.0):
    z = mat[np.isfinite(mat)]
    if z.size == 0:
        return 1.0
    t = float(np.nanpercentile(np.abs(z), pct))
    return t if t > 0 else 1.0


thr = max(_sym_lim(SNR_E), _sym_lim(SNR_B))

fig, axes = plt.subplots(1, 2, figsize=(10, 5), constrained_layout=True)
im0 = axes[0].imshow(
    np.flipud(SNR_E),
    vmin=-thr,
    vmax=thr,
    cmap="RdBu_r",
    extent=[np.min(x_bin), np.max(x_bin), np.min(y_bin), np.max(y_bin)],
    aspect="auto",
)
axes[0].set_xlabel("x [virtual pix]")
axes[0].set_ylabel("y [virtual pix]")
axes[0].set_title(r"E-mode $M_{\rm ap}$ S/N")

im1 = axes[1].imshow(
    np.flipud(SNR_B),
    vmin=-thr,
    vmax=thr,
    cmap="RdBu_r",
    extent=[np.min(x_bin), np.max(x_bin), np.min(y_bin), np.max(y_bin)],
    aspect="auto",
)
axes[1].set_xlabel("x [virtual pix]")
axes[1].set_ylabel("y [virtual pix]")
axes[1].set_title(r"B-mode $M_{\rm ap}$ S/N")

fig.colorbar(im1, ax=axes, shrink=0.85, label="S/N")
fig.suptitle(
    rf"Tract {CONFIG['TRACT']} | Schirmer $R_s$={Rs_input:.0f} pix | bin={bin_size:.0f} pix | $|S/N|_{{95}}$={thr:.2f}",
    y=1.02,
)
plt.show()


## 7.0 Optional: global peak in E-mode S/N (script-style)


In [ ]:
mat = SNR_E
max_val = float(np.nanmax(mat[np.isfinite(mat)]))
indices = np.where(mat == max_val)
if indices[0].size:
    row_peak, col_peak = int(indices[0][0]), int(indices[1][0])
    x_out = x_min + col_peak * bin_size + 0.5 * bin_size
    y_out = y_min + row_peak * bin_size + 0.5 * bin_size
    print("Peak E-mode S/N:", max_val, "at virtual (x, y) =", (x_out, y_out))
else:
    print("No finite peak")
